# NVIDIA Nemotron Reasoning Challenge — Anaconda Cloud

**Note:** Anaconda Cloud no longer offers GPU runtimes. Use this notebook to **download data** and run **EDA + data prep** (CPU only). For **training**, use [Kaggle](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge) with `kaggle_notebook.ipynb` (free GPU).

## Workflow

| Step              | Anaconda (this notebook) | Kaggle                          |
|-------------------|--------------------------|----------------------------------|
| Download data     | ✓ via Kaggle API         | ✓ add competition dataset       |
| EDA + data prep   | ✓                        | ✓                               |
| **LoRA training** | ✗ (no GPU)               | ✓ 2× T4 GPU                     |
| Package           | ✓ (with pre-trained adapter) | ✓                          |

**Option A:** Run this notebook for data prep, then copy `data/train_sft.jsonl` and project files to Kaggle and run training there.

**Option B:** Run the full pipeline on Kaggle (e.g. `kaggle_notebook.ipynb`) — add competition data + Nemotron model dataset.

In [ ]:
# ========== CONFIG ==========
# Path to data folder (train.csv, test.csv)
DATA_DIR = "data"

# Kaggle API (to download competition data + model). Get credentials from:
#   Kaggle → Account → Create New API Token (downloads kaggle.json)
# Paste the values here, or set env vars KAGGLE_USERNAME / KAGGLE_KEY
KAGGLE_USERNAME = ""  # e.g. "your-kaggle-username"
KAGGLE_KEY = ""      # e.g. "abc123..." from kaggle.json

# If True, download competition data + model from Kaggle (requires KAGGLE_USERNAME/KEY)
USE_KAGGLE_DOWNLOAD = True  # Set False if you uploaded data/model manually

# GitHub repo (used to clone project if scripts/ not found)
GITHUB_REPO = "https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge"

USE_SYNTHETIC_ONLY = True
SKIP_EVAL = True
USE_PEFT_ONLY = True

# Skip training on Anaconda (no GPU runtimes). Set False only if you have a GPU runtime.
SKIP_TRAIN = True  # Run training on Kaggle instead
SKIP_PACKAGE = True  # submission.zip needs lora_adapter from training

# Local Nemotron model (set by Kaggle download, or specify if you have it)
NEMOTRON_INPUT_PATH = ""

# Use lighter deps when skipping train (no mamba-ssm, bitsandbytes)
PIP_REQUIREMENTS_FILE = "requirements-local.txt" if SKIP_TRAIN else "requirements-kaggle-peft.txt"
SKIP_PIP_INSTALL = False

## 0. Connect to Kaggle and download data + model

Downloads competition data (train.csv, test.csv) and the Nemotron model from Kaggle.

**Requirements:**
- Kaggle account: [kaggle.com](https://kaggle.com) → sign up
- Accept competition rules: [competition page](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge) → "Join Competition"
- API token: Kaggle → Profile → Account → **Create New API Token** → copy `username` and `key` from `kaggle.json` into the config cell above

In [ ]:
import os
import subprocess
import sys
import zipfile

# Project root
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, DATA_DIR) if not os.path.isabs(DATA_DIR) else DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)

if not USE_KAGGLE_DOWNLOAD:
    print("USE_KAGGLE_DOWNLOAD=False — skipping. Ensure train.csv, test.csv and model are in place.")
else:
    # Set Kaggle credentials
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        os.environ["KAGGLE_KEY"] = KAGGLE_KEY
    elif not os.path.isfile(os.path.expanduser("~/.kaggle/kaggle.json")):
        raise RuntimeError(
            "Kaggle credentials not set. Either:\n"
            "  1. Set KAGGLE_USERNAME and KAGGLE_KEY in the config cell, or\n"
            "  2. Upload ~/.kaggle/kaggle.json to your project"
        )

    # Install kaggle + kagglehub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle", "kagglehub"], check=True)

    # Download competition data
    train_csv = os.path.join(DATA_DIR, "train.csv")
    test_csv = os.path.join(DATA_DIR, "test.csv")
    if not os.path.isfile(train_csv) or not os.path.isfile(test_csv):
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        print("Downloading competition data...")
        api.competition_download_files("nvidia-nemotron-model-reasoning-challenge", path=PROJECT_ROOT)
        zip_path = os.path.join(PROJECT_ROOT, "nvidia-nemotron-model-reasoning-challenge.zip")
        if os.path.isfile(zip_path):
            import shutil
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(PROJECT_ROOT)
            for f in ["train.csv", "test.csv"]:
                for root, _, files in os.walk(PROJECT_ROOT):
                    if f in files:
                        shutil.copy2(os.path.join(root, f), os.path.join(DATA_DIR, f))
                        break
            os.remove(zip_path)
        print("Competition data ready.")
    else:
        print("Competition data already present.")

    # Download Nemotron model (skip if SKIP_TRAIN — tokenizer loads from HF for data prep)
    if not SKIP_TRAIN and (not NEMOTRON_INPUT_PATH or not os.path.isdir(NEMOTRON_INPUT_PATH)):
        model_path = None
        try:
            import kagglehub
            print("Downloading Nemotron model from Kaggle Model Hub (~5GB)...")
            model_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
        except Exception as e:
            print(f"kagglehub: {e}. Trying Kaggle dataset...")
            try:
                from kaggle.api.kaggle_api_extended import KaggleApi
                api = KaggleApi()
                api.authenticate()
                model_dir = os.path.join(PROJECT_ROOT, "nemotron_model")
                os.makedirs(model_dir, exist_ok=True)
                api.dataset_download_files("kishanvavdara/nvidia-nemotron-3-nano-30b-a3b-bf16", path=model_dir)
                zip_path = next((os.path.join(model_dir, f) for f in os.listdir(model_dir) if f.endswith(".zip")), None)
                if zip_path and os.path.isfile(zip_path):
                    with zipfile.ZipFile(zip_path, "r") as z:
                        z.extractall(model_dir)
                    for root, _, files in os.walk(model_dir):
                        if "config.json" in files:
                            model_path = root
                            break
            except Exception as e2:
                print(f"Dataset fallback: {e2}")
        if model_path and os.path.isfile(os.path.join(model_path, "config.json")):
            NEMOTRON_INPUT_PATH = model_path
            os.environ["NEMOTRON_MODEL_PATH"] = model_path
            os.environ["HF_HUB_OFFLINE"] = "1"
            print(f"Model ready at: {model_path}")
        else:
            print("Model not found via Kaggle. Pipeline will try Hugging Face (needs ~5GB download).")

## 1. Setup project and data

In [ ]:
import os
import shutil
import subprocess

# Project root = directory containing this notebook
PROJECT_ROOT = os.getcwd()
if not os.path.isabs(DATA_DIR):
    DATA_DIR = os.path.join(PROJECT_ROOT, DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

# HF cache in project to avoid filling home
_hf = os.path.join(PROJECT_ROOT, ".hf")
os.makedirs(_hf, exist_ok=True)
os.environ.setdefault("HF_HOME", _hf)
os.environ.setdefault("TRANSFORMERS_CACHE", os.path.join(_hf, "hub"))

# Clone project if scripts/ missing
if not os.path.isfile(os.path.join(PROJECT_ROOT, "scripts", "01_eda.py")):
    if GITHUB_REPO:
        repo = GITHUB_REPO.strip().rstrip("/").replace(".git", "")
        if not repo.endswith(".git"):
            repo += ".git"
        print(f"Cloning {repo}...")
        subprocess.run(["git", "clone", "--depth", "1", repo, "repo_temp"], cwd=PROJECT_ROOT, check=True)
        for item in os.listdir(os.path.join(PROJECT_ROOT, "repo_temp")):
            if item in ("data", ".git"):
                continue
            src = os.path.join(PROJECT_ROOT, "repo_temp", item)
            dst = os.path.join(PROJECT_ROOT, item)
            if os.path.exists(dst):
                shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
            shutil.move(src, dst)
        shutil.rmtree(os.path.join(PROJECT_ROOT, "repo_temp"), ignore_errors=True)
        print("Clone done.")
    else:
        raise FileNotFoundError("scripts/ not found. Upload the project or set GITHUB_REPO.")

# Check data
train_csv = os.path.join(DATA_DIR, "train.csv")
test_csv = os.path.join(DATA_DIR, "test.csv")
if not os.path.isfile(train_csv) or not os.path.isfile(test_csv):
    raise FileNotFoundError(
        f"Missing train.csv or test.csv in {DATA_DIR}/\n"
        "Download from: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/data"
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print("Data files:", os.listdir(DATA_DIR))

## 2. Install dependencies

In [ ]:
import sys

if not SKIP_PIP_INSTALL:
    req_path = os.path.join(PROJECT_ROOT, PIP_REQUIREMENTS_FILE)
    if not os.path.isfile(req_path):
        raise FileNotFoundError(f"Missing {PIP_REQUIREMENTS_FILE}")
    print("Installing requirements...")
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "-r", req_path], cwd=PROJECT_ROOT).returncode
    if rc != 0:
        raise RuntimeError("pip install failed")
    if not SKIP_TRAIN:
        print("Installing mamba-ssm, causal-conv1d (Nemotron)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mamba-ssm", "causal-conv1d", "--force-reinstall"], cwd=PROJECT_ROOT)
    print("Done.")
else:
    print("SKIP_PIP_INSTALL=True")

## 3. Run pipeline

In [ ]:
import sys

def _find_model_dir(base):
    if not base or not os.path.isdir(base):
        return None
    if os.path.isfile(os.path.join(base, "config.json")):
        return base
    for name in os.listdir(base):
        sub = os.path.join(base, name)
        if os.path.isdir(sub) and os.path.isfile(os.path.join(sub, "config.json")):
            return sub
    return None

# Model path: local or HF
if NEMOTRON_INPUT_PATH and os.path.isdir(NEMOTRON_INPUT_PATH):
    path = _find_model_dir(NEMOTRON_INPUT_PATH) or NEMOTRON_INPUT_PATH
    if os.path.isfile(os.path.join(path, "config.json")):
        os.environ["NEMOTRON_MODEL_PATH"] = path
        os.environ["HF_HUB_OFFLINE"] = "1"
        print(f"Using local model: {path}")

args = [sys.executable, "run_all.py"]
if USE_SYNTHETIC_ONLY:
    args.append("--synthetic-only")
if SKIP_EVAL:
    args.append("--skip-eval")
if SKIP_TRAIN:
    args.append("--skip-train")
if SKIP_PACKAGE:
    args.append("--skip-package")
if USE_PEFT_ONLY:
    args.append("--peft-only")

print("Running:", " ".join(args))
result = subprocess.run(args, cwd=PROJECT_ROOT, env={**os.environ, "PYTHONPATH": PROJECT_ROOT})
if result.returncode != 0:
    sys.exit(result.returncode)
print("Pipeline finished.")

## 4. Verify output

In [ ]:
if SKIP_TRAIN:
    sft_path = os.path.join(PROJECT_ROOT, "data", "train_sft.jsonl")
    if os.path.isfile(sft_path):
        print(f"train_sft.jsonl ready: {os.path.getsize(sft_path):,} bytes")
        print("Next: Run training on Kaggle (kaggle_notebook.ipynb) with this data.")
    else:
        print("train_sft.jsonl not found. Check pipeline output above.")
else:
    sub_path = os.path.join(PROJECT_ROOT, "submission.zip")
    if os.path.isfile(sub_path):
        print(f"submission.zip: {os.path.getsize(sub_path):,} bytes — submit to Kaggle.")
    else:
        print("submission.zip not found.")